In [1]:
import yaml
import json
from app.datasets.loader import load_multiple_test_cases
from app.datasets.validator import validate_dataset_schema
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests
from app.utils.db import save_results_on_cosmos

In [ ]:
file_list = [
  './app/data/raw/tramites.xlsx',
  './app/data/raw/accesibilidad.xlsx',
  './app/data/raw/descubrir.xlsx',
  './app/data/raw/solicitudes.xlsx',
  './app/data/raw/organigrama.xlsx'
]

test_config = {
  'TIMINGS': {'test': True, 'report': False},
  'TOKENS': {'test': True, 'report': False},
  'FOUNDRYS': {'test': True, 'report': False},
  'TRIAGE': {'test': True, 'report': False},
  'ROUTER': {'test': True, 'report': False},
  'GROUNDING': {'test': True, 'report': False},
  'SAVE_RESULTS': False,
  # 'PATH': './app/data/processed/reports/report'
}   

df = load_multiple_test_cases(file_list)
df = validate_dataset_schema(df)

with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)

In [3]:
# responses = client.query_batch(df['user_input'],df['reference'])

In [4]:
# save_responses_in_json, response_file_path = client.save_api_responses(responses)

response_file_path = './app/data/processed/outcome_20260205-120729.json'

In [5]:
with open(response_file_path, 'r', encoding='UTF-8') as f:
  data = json.load(f)

results = run_tests(
  config = test_config, 
  data = data, 
  df = df, 
  timestamp = str(response_file_path)
)

In [6]:
print(results)

{'timestamp': '20260205-120729', 'nodes': {'triage': {'positives': 802, 'total': 884, 'result': 90.72}, 'router': {'positives': 753, 'total': 802, 'result': 93.89}, 'grounding': {'positives': 404, 'total': 412, 'result': 98.06}}, 'timings': {'reformulate': {'prom': np.float64(1.284), 'p90': np.float64(1.844), 'p95': np.float64(2.171), 'quantity': 879}, 'triage': {'prom': np.float64(1.62), 'p90': np.float64(2.363), 'p95': np.float64(2.967), 'quantity': 879}, 'router': {'prom': np.float64(1.488), 'p90': np.float64(2.139), 'p95': np.float64(2.606), 'quantity': 879}, 'ag_call': {'prom': np.float64(4.294), 'p90': np.float64(6.227), 'p95': np.float64(7.611), 'quantity': 412}, 'personality': {'prom': np.float64(3.502), 'p90': np.float64(4.899), 'p95': np.float64(5.669), 'quantity': 412}, 'grounding': {'prom': np.float64(2.232), 'p90': np.float64(3.122), 'p95': np.float64(3.701), 'quantity': 412}, 'retriever': {'prom': np.float64(0.602), 'p90': np.float64(0.661), 'p95': np.float64(1.516), 'qua

In [7]:
# if test_config.get('SAVE_RESULTS'):
#   save_results_on_cosmos(results=results)